# Video Object Tracking with lazysammy2

This notebook demonstrates the **video tracking capabilities** of `lazysammy2`, a simplified wrapper around Meta's [SAM 2](https://github.com/facebookresearch/sam2).

You will learn how to:
1. Load a video — either a **video file** (mp4, avi, …) or a **folder of frames**
2. **Interactively pick prompts** (points & boxes) by clicking on frames
3. Track multiple objects simultaneously with prompts at arbitrary frames
4. Propagate segmentations forward, backward, and bidirectionally
5. Inspect, visualise, and save results
6. Manage objects (add / remove mid-sequence, reset)

> **Requirements:** Install the package first with `uv sync --extra viz` to include matplotlib visualisation support.

## 0 — Setup

In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from lazysammy2 import (
    SAM2,
    PromptPicker,
    VideoTracker,
    draw_masks_on_image,
    extract_frames,
    pick_box_on_image,
    pick_points_on_image,
    preview_frame,
    save_video_overlay,
    show_video_frame,
)

%matplotlib widget
# %matplotlib inline
plt.rcParams["figure.dpi"] = 120

## 1 — Load a video

`lazysammy2` accepts **both video files and frame directories**.
When you pass a video file (`.mp4`, `.avi`, `.mov`, …), frames are
automatically extracted for you.

In [ ]:
# === Option A: Pass a video file directly ===
# Frames are extracted automatically into a sibling folder.
# VIDEO_PATH = "your_video.mp4"

# === Option B: Use a pre-existing frames directory ===
# VIDEO_PATH = "path/to/frames/"

# === Option C: Extract frames manually with more control ===
# frames_dir = extract_frames(
#     "your_video.mp4",
#     output_dir="my_frames/",   # optional, defaults to <video>_frames/
#     every_n=2,                 # keep every 2nd frame
#     max_frames=200,            # stop after 200 frames
# )
# VIDEO_PATH = frames_dir

# --- Set your path here ---
VIDEO_PATH = "assets/fair.mp4"  # <-- change this

In [ ]:
# Initialise SAM2 and start a session.
# If VIDEO_PATH is a video file, frames are extracted automatically.
sam = SAM2("large")

session = sam.video(
    VIDEO_PATH,
    # --- Only used when VIDEO_PATH is a video file: ---
    # every_n=1,        # keep every frame
    # max_frames=None,   # no limit
    # frames_dir=None,   # auto: <video>_frames/
)
print(f"Session ready: {session.num_frames} frames from {session.video_dir}")

## 2 — Preview frames & read coordinates

Before adding prompts you need to know *where* to click.
Use `preview_frame()` to display a frame with a coordinate grid.

With `%matplotlib widget`, you can **hover** to see live `(x, y)` in the toolbar.
With `%matplotlib inline`, the grid lines and tick labels still help you estimate coordinates.

In [ ]:
# Show frame 0 with a coordinate grid
# With %matplotlib widget: hover to read (x, y) in the toolbar
# With %matplotlib inline: use the grid lines to estimate coordinates
preview_frame(session.video_dir, frame_idx=0, grid_step=100)

In [ ]:
# You can also load a frame as a numpy array
first_frame = session.get_frame(0)
print(f"Frame shape: {first_frame.shape}  (H, W, C)")

## 3 — Interactive prompt placement with `PromptPicker`

Specifying exact `(x, y)` coordinates by hand is tricky.
The **`PromptPicker`** lets you **click directly on the frame** to add
points or draw boxes, then automatically sends the prompt to the session.

> **Important:** Interactive clicking requires `%matplotlib widget` (set in cell 2 above).
> If you're using `%matplotlib inline`, the plot will display but clicking won't register —
> use the manual API in section 3.3 instead (coordinates from `preview_frame`'s grid).

| Action | How |
|--------|-----|
| Foreground point | **Left-click** |
| Background point | **Right-click** |
| Finish picking | **Enter** or close the figure |
| Draw a box | **Left-click** top-left, then **left-click** bottom-right |

In [ ]:
# Create a PromptPicker bound to the session
picker = PromptPicker(session)

# Preview frame 0 with coordinate grid
picker.preview(frame_idx=0)

In [ ]:
# --- Interactive: click to place points for object 1 ---
# Left-click  = foreground point
# Right-click = background point
# Press Enter or middle-click to finish
points, labels = picker.add_points_interactive(frame_idx=0, obj_id=1)
print(f"Added points: {points}, labels: {labels}")

In [ ]:
plt.close()

### 3.2 — Interactive box prompt

In [ ]:
# --- Interactive: draw a bounding box for object 2 ---
# Click top-left corner, then bottom-right corner
box = picker.add_box_interactive(frame_idx=0, obj_id=2)
print(f"Added box: {box}")

In [ ]:
# Show both objects on frame 0
fm = session.add_points(frame_idx=0, obj_id=1, points=points, labels=labels)  # re-fetch frame masks
fig = show_video_frame(first_frame, fm, alpha=0.5)
plt.show()

### 3.3 — Prompts on later frames

You can add prompts on **any frame**, not just the first. This is useful when:
- An object enters the scene mid-video
- You want to correct a drifted mask

Both the `PromptPicker` and manual API accept any `frame_idx`.

In [ ]:
# Add a third object starting at frame 10 — interactively or manually
# Interactive:
# picker.add_points_interactive(frame_idx=10, obj_id=3)

# Manual:
frame_result = session.add_points(
    frame_idx=10,
    obj_id=3,
    points=[[350, 250]],
    labels=[1],
)
print(f"Added object 3 on frame 10: {frame_result.object_ids}")

### 3.4 — Combining foreground and background points

You can supply **negative points** (label=0) to exclude regions from the mask.
When using `PromptPicker`, **left-click = foreground**, **right-click = background** — no need to manually specify labels.

In [ ]:
# Refine object 1 on frame 0 with an additional positive + negative point
frame_result = session.add_points(
    frame_idx=0,
    obj_id=1,
    points=[
        [200, 300],   # foreground
        [210, 280],   # foreground
        [180, 350],   # background — exclude this region
    ],
    labels=[1, 1, 0],
    clear_old=True,   # replace previous points (default)
)
print(f"Refined mask for object 1, area: {frame_result.masks[0].sum()} px")

## 4 — Propagate through the video

Once prompts are set, call `propagate()` to track objects across all frames.

In [ ]:
# Forward propagation (from earliest prompted frame to the end)
results = session.propagate()

print(f"Tracked {len(results)} frames")
print(f"Object IDs: {results.object_ids}")

In [ ]:
session.save_overlay("output/tracking.mp4", fps=24, alpha=0.5)

In [ ]:
# Inspect a few frames
sample_frames = [0, len(results) // 4, len(results) // 2, 3 * len(results) // 4]
sample_frames = [f for f in sample_frames if f in results.frame_masks]

fig, axes = plt.subplots(1, len(sample_frames), figsize=(5 * len(sample_frames), 5))
if len(sample_frames) == 1:
    axes = [axes]

for ax, fidx in zip(axes, sample_frames):
    fm = results[fidx]
    frame_img = cv2.cvtColor(cv2.imread(str(frame_files[fidx])), cv2.COLOR_BGR2RGB)
    overlay = draw_masks_on_image(frame_img, fm.masks, alpha=0.5)
    ax.imshow(overlay)
    ax.set_title(f"Frame {fidx} ({len(fm.object_ids)} obj)")
    ax.axis("off")

plt.tight_layout()
plt.show()

### 4.1 — Bidirectional propagation

If your prompt is on a mid-video frame, use `propagate_bidirectional()` to track **both forward and backward**.

In [ ]:
# Reset and demonstrate bidirectional tracking from a mid-video prompt
session.reset()

mid = session.num_frames // 2
mid_frame = cv2.cvtColor(cv2.imread(str(frame_files[mid])), cv2.COLOR_BGR2RGB)

# Prompt object on the middle frame
session.add_points(
    frame_idx=mid,
    obj_id=1,
    points=[[200, 300]],
    labels=[1],
)

# Propagate in both directions
results_bidir = session.propagate_bidirectional()
print(f"Bidirectional: tracked {len(results_bidir)} frames from mid-frame {mid}")

### 4.2 — Partial propagation

You can limit propagation to a certain number of frames.

In [ ]:
session.reset()

session.add_points(frame_idx=0, obj_id=1, points=[[200, 300]], labels=[1])

# Only propagate for 30 frames
partial = session.propagate(max_frames=30)
print(f"Partial propagation: {len(partial)} frames")

## 5 — Inspect results in detail

The `VideoResults` object supports iteration, indexing, and per-object queries.

In [ ]:
# Re-run full tracking with two objects for the rest of the examples
session.reset()
session.add_points(frame_idx=0, obj_id=1, points=[[200, 300]], labels=[1])
session.add_box(frame_idx=0, obj_id=2, box=[400, 150, 600, 400])
results = session.propagate()

# --- Iterate over all frames ---
for fm in results:
    # fm is a FrameMasks object
    pass  # process each frame

# --- Access a single frame ---
fm0 = results[0]
print(f"Frame 0 objects: {fm0.object_ids}, mask shape: {fm0.masks.shape}")

# --- Get all masks for one object across time ---
obj1_masks = results.get_object_masks(obj_id=1)
print(f"Object 1 tracked in {len(obj1_masks)} frames")

# --- List all tracked object ids ---
print(f"All object IDs: {results.object_ids}")

In [ ]:
# Plot mask area over time for each object
fig, ax = plt.subplots(figsize=(12, 4))

for obj_id in results.object_ids:
    obj_masks = results.get_object_masks(obj_id)
    frames_sorted = sorted(obj_masks.keys())
    areas = [obj_masks[f].sum() for f in frames_sorted]
    ax.plot(frames_sorted, areas, label=f"Object {obj_id}")

ax.set_xlabel("Frame index")
ax.set_ylabel("Mask area (pixels)")
ax.set_title("Object mask area over time")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6 — Manage objects mid-sequence

You can dynamically add, remove, and correct objects during a session.

In [ ]:
# Remove object 2 and re-propagate
session.remove_object(obj_id=2)
results_after_removal = session.propagate()
print(f"After removal — Object IDs: {results_after_removal.object_ids}")

In [ ]:
# Clear prompts for a specific object on a specific frame
# Useful for correcting a bad prompt without removing the entire object
session.clear_frame_prompts(frame_idx=0, obj_id=1)

# Re-add with better coordinates
session.add_points(
    frame_idx=0,
    obj_id=1,
    points=[[210, 310], [220, 290]],
    labels=[1, 1],
)
results_corrected = session.propagate()
print(f"Corrected tracking: {len(results_corrected)} frames")

In [ ]:
# Full reset — clears everything, ready for a fresh start
session.reset()
print("Session reset. All prompts and state cleared.")

## 7 — Save results

Results can be saved in three formats: **PNG** masks, **NumPy** `.npy` arrays, or **COCO RLE** JSON.

In [ ]:
# Re-run tracking for saving
session.add_points(frame_idx=0, obj_id=1, points=[[200, 300]], labels=[1])
session.add_box(frame_idx=0, obj_id=2, box=[400, 150, 600, 400])
results = session.propagate()

In [ ]:
# Save as PNG masks (one per object per frame)
session.save("output/masks_png", fmt="png")

# Save as NumPy arrays
# session.save("output/masks_npy", fmt="npy")

# Save as COCO RLE JSON
# session.save("output/masks_rle", fmt="coco_rle")

print("Saved masks in all three formats!")

In [ ]:
# Save visualisation overlays (PNG frames with masks drawn on top)
save_video_overlay(FRAMES_DIR, results, "output/overlay_frames", alpha=0.5)
print("Overlay frames saved to output/overlay_frames/")

## 8 — Using the mask prompt

Besides points and boxes, you can initialise tracking from an **existing binary mask** —
for example from another segmentation model or manual annotation.

In [ ]:
# Direct VideoTracker usage (equivalent to sam.video(...))
tracker = VideoTracker(
    "large",
    device="cuda",          # explicit device
    vos_optimized=False,     # set True for torch.compile speedup (CUDA only)
)

# Accepts a video file OR a frames directory, same as sam.video()
session2 = tracker.new_session(
    VIDEO_PATH,
    offload_video_to_cpu=True,   # reduce GPU memory usage
    offload_state_to_cpu=False,
    every_n=1,                   # frame subsampling (video file only)
)

session2.add_points(frame_idx=0, obj_id=1, points=[[200, 300]], labels=[1])
results2 = session2.propagate()
print(f"Direct tracker: {len(results2)} frames, device={tracker.device}")

## 9 — Standalone interactive helpers (no session needed)

You can use `pick_points_on_image` and `pick_box_on_image` on **any image** —
they return coordinates you can later pass to the session manually.

> These also require `%matplotlib widget` for the clicks to register.

In [ ]:
# Pick points on any image (returns coords, doesn't need a session)
some_image = session.get_frame(0)
points, labels = pick_points_on_image(some_image, n=3)
print(f"Picked: {points}, {labels}")

# Draw a box on any image
box = pick_box_on_image(some_image)
print(f"Box: {box}")

# Then use them with the session:
# session.add_points(frame_idx=0, obj_id=1, points=points, labels=labels)
# session.add_box(frame_idx=0, obj_id=2, box=box)

## 11 — Memory tips

| Tip | How |
|-----|-----|
| Use a smaller model | `SAM2("tiny")` or `SAM2("small")` |
| Offload frames to CPU | `sam.video(path, offload_video_to_cpu=True)` |
| Offload state to CPU | `sam.video(path, offload_state_to_cpu=True)` |
| Subsample frames | `sam.video("video.mp4", every_n=3)` |
| Limit propagation | `session.propagate(max_frames=100)` |
| Free the model | `del sam` or restart the kernel |

For the fastest inference on CUDA, enable `vos_optimized=True` when constructing `SAM2` or `VideoTracker`.

---

**That's it!** You now know how to:
- Load a video file or frame directory
- **Interactively** pick points & boxes with `PromptPicker`
- Track multiple objects through video
- Save results in PNG, NumPy, or COCO RLE

For image segmentation and auto-mask examples, see the package README or run:
```python
pred = sam.segment("photo.jpg", points=[[x, y]], labels=[1])
auto = sam.auto_segment("photo.jpg")
```